In [1]:
import minsearch
import json
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [4]:
# flatten the json data
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [5]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [6]:
index.fit(docs=documents)

In [7]:
def search(query, filter_dict=None, boost_dict=None, n=5):
    results=index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=n
    )

    return results

In [8]:
def build_prompt(query, search_results):
    prompt_template = """
        You are a teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
        Use only the facts from the context when answering the QUESTION.
        If the context does not contain the answer, output NONE.

        QUESTION: {question}
        CONTEXT: {context}

    """.strip()

    context = ""

    for doc in search_results:
        context += f"section {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    
    return prompt

In [9]:
def llm(client, prompt):
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.output_text

In [15]:
def rag(client, query, n):
    results = search(query=query, n=n)
    prompt = build_prompt(query=query, search_results=results)
    answer = llm(client=client, prompt=prompt)

    return answer

In [16]:
query = "the course just started. Can I still enroll?"
client = OpenAI()
rag(client=client, query=query, n=20)

'Yes, you can still enroll in the course after it has started. You won’t be able to submit some of the homework, but you can participate and work on projects to be eligible for a certificate.'